# 10 — LangGraph Workflow

**Objective**: build and run the full `StateGraph` (Section 7) end to end for the first time — every node wired to the real connectors/services built in Phases 3-7, no shortcuts. Confirm it reproduces the exact RAG distribution every earlier phase computed independently by hand or ad hoc script.

**Dependencies**: `src/graph/nodes.py`, `src/graph/workflow.py`, `src/graph/state.py`.

**Configuration**: `NodeDeps` bundles the live connector/service instances a compiled graph needs — build it once, `build_graph(deps)` compiles a graph you can `.invoke()` as many times as you want with different `requested_at`/`user_question` values.

In [1]:
import os
import sys
import shutil
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from datetime import datetime, timezone

from src.connectors.jira_client import build_default_jira_client
from src.connectors.financial_client import CSVFinancialDataSource
from src.services import project_unifier
from src.services.memory_store import FileMemoryStore
from src.graph.nodes import NodeDeps
from src.graph.workflow import build_graph

NOTEBOOK_SNAPSHOT_DIR = PROJECT_ROOT / "data/snapshots"
shutil.rmtree(NOTEBOOK_SNAPSHOT_DIR, ignore_errors=True)  # clean slate for a reproducible run

deps = NodeDeps(
    jira_client=build_default_jira_client(),
    financial_source=CSVFinancialDataSource(),
    memory_store=FileMemoryStore(base_dir=NOTEBOOK_SNAPSHOT_DIR),
    mapping=project_unifier.load_project_mapping(),
)
graph = build_graph(deps)
print("Graph compiled.")

Graph compiled.


## The graph's shape

In [2]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify_request(classify_request)
	fetch_delivery_data(fetch_delivery_data)
	validate_delivery_data(validate_delivery_data)
	fetch_financial_data(fetch_financial_data)
	validate_financial_data(validate_financial_data)
	unify_projects(unify_projects)
	retrieve_historical_memory(retrieve_historical_memory)
	calculate_metrics(calculate_metrics)
	analyze_delivery_risk(analyze_delivery_risk)
	analyze_financial_risk(analyze_financial_risk)
	analyze_cross_domain_risk(analyze_cross_domain_risk)
	validate_findings(validate_findings)
	generate_response(generate_response)
	persist_snapshot(persist_snapshot)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify_request;
	analyze_cross_domain_risk --> validate_findings;
	analyze_delivery_risk --> analyze_financial_risk;
	analyze_financial_risk --> analyze_cross_domain_risk;
	calculate_metrics --> analyze_delivery_risk;
	classify_request --> fetch_d

## Portfolio-wide run

In [3]:
result = graph.invoke({
    "user_question": "What is the status of our portfolio?",
    "request_id": "req-001",
    "requested_at": datetime(2026, 9, 15, tzinfo=timezone.utc),
})

print(result["final_answer"])

{"request_id": "req-001", "timestamp": "2026-09-03T03:22:55.397402+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "req-001", "timestamp": "2026-09-03T03:22:55.397822+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.02}
{"request_id": "req-001", "timestamp": "2026-09-03T03:22:55.398375+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "req-001", "timestamp": "2026-09-03T03:22:55.416324+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 270, "sprint_count": 20, "partial_failure": false}
{"request_id": "req-001", "timestamp": "2026-09-03T03:22:55.416401+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 17.98}
{"request_id": "req-001", "timestamp": "2026-09-03T03:22:55.416918+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "req-001", "timestamp": "2026-09-03T03:22:55.417015+00:00", "event": "graph_node_end", 

## Cross-check against every earlier phase

Phase 1 hand-predicted this RAG distribution; Phase 6/7/8 have each independently reproduced it through progressively more real code. This is the fourth time, now through the actual orchestrated graph.

In [4]:
expected = {
    "PROJECT-10001": "RED", "PROJECT-10002": "RED", "PROJECT-10003": "RED",
    "PROJECT-10004": "RED", "PROJECT-10005": "RED",
    "PROJECT-10006": "AMBER", "PROJECT-10007": "AMBER",
}
actual = {p.project_id: p.risk_status.value for p in result["unified_projects"]}
for pid, expected_rag in expected.items():
    status = "OK" if actual[pid] == expected_rag else "MISMATCH"
    print(f"  {pid:16s} expected={expected_rag:6s} actual={actual[pid]:6s}  [{status}]")
assert actual == expected

  PROJECT-10001    expected=RED    actual=RED     [OK]
  PROJECT-10002    expected=RED    actual=RED     [OK]
  PROJECT-10003    expected=RED    actual=RED     [OK]
  PROJECT-10004    expected=RED    actual=RED     [OK]
  PROJECT-10005    expected=RED    actual=RED     [OK]
  PROJECT-10006    expected=AMBER  actual=AMBER   [OK]
  PROJECT-10007    expected=AMBER  actual=AMBER   [OK]


## Why confidence is LOW for this run

Not a bug — this dataset genuinely has stale Jira data (relative to `requested_at`) and the known financial reconciliation mismatch (Phase 4) on every mapped project. `validate_findings` correctly reflects that instead of hiding it.

In [5]:
print(f"confidence: {result['confidence']}")
print(f"validation_passed: {result['validation_passed']}")
print()
data_quality = [r for r in result["risks"] if r.category == "Data Quality"]
print(f"{len(data_quality)} data-quality findings:")
for r in data_quality[:6]:
    print(f"  [{r.severity.value}] {r.project_id}: {r.description}")

confidence: LOW CONFIDENCE
validation_passed: True

9 data-quality findings:
  [MEDIUM] PORTFOLIO: Jira data is 284.6 hours old, exceeding the 24h threshold
  [MEDIUM] PROJECT-10001: Source-reported remaining budget does not match this pipeline's computed figure
  [MEDIUM] PROJECT-10002: Source-reported remaining budget does not match this pipeline's computed figure
  [MEDIUM] PROJECT-10003: Source-reported remaining budget does not match this pipeline's computed figure
  [MEDIUM] PROJECT-10004: Source-reported remaining budget does not match this pipeline's computed figure
  [MEDIUM] PROJECT-10005: Source-reported remaining budget does not match this pipeline's computed figure


## Scoped query: project_filter narrows every downstream node

In [6]:
scoped = graph.invoke({
    "user_question": "Why is Phoenix Platform Modernization at risk?",
    "request_id": "req-002",
    "requested_at": datetime(2026, 9, 15, tzinfo=timezone.utc),
})
print(f"intent={scoped['intent']}  project_filter={scoped['project_filter']}")
print(f"jira_issues fetched: {len(scoped['jira_issues'])}  (37 = just PHX, not the full 270-issue portfolio)")
print(f"unified_projects: {[p.project_id for p in scoped['unified_projects']]}")

{"request_id": "req-002", "timestamp": "2026-09-03T03:22:55.481553+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "req-002", "timestamp": "2026-09-03T03:22:55.481780+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
{"request_id": "req-002", "timestamp": "2026-09-03T03:22:55.482144+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "req-002", "timestamp": "2026-09-03T03:22:55.484595+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 37, "sprint_count": 3, "partial_failure": false}
{"request_id": "req-002", "timestamp": "2026-09-03T03:22:55.484646+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 2.47}
{"request_id": "req-002", "timestamp": "2026-09-03T03:22:55.485028+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "req-002", "timestamp": "2026-09-03T03:22:55.485113+00:00", "event": "graph_node_end", "no

## Evidence and citations for one project

In [7]:
phx_risks = [r for r in scoped["risks"] if r.project_id == "PROJECT-10001" and r.category in ("Delivery", "Financial")]
for r in phx_risks:
    print(f"[{r.category}] severity={r.severity.value}")
    for line in r.evidence:
        print(f"    - {line}")

print()
print("Citations:")
for c in scoped["citations"]:
    print(f"  {c['source_system']}: {c['source_record_id']} (retrieved {c['retrieved_timestamp']})")

[Delivery] severity=HIGH
    - PHX-17: blocked 240 days (Flagged blocked via blocker_status field (source provided no reason text))
    - PHX-4: blocked 235 days (Flagged blocked via blocker_status field (source provided no reason text))
    - PHX-15: blocked 224 days (Unresolved dependency: PHX-3)
    - PHX-26: blocked 224 days (Flagged blocked via blocker_status field (source provided no reason text))
    - PHX-6 depends on PHX-4: not done (status=IN_REVIEW)
    - PHX-14 depends on PHX-2: not done (status=IN_PROGRESS)
    - PHX-22 depends on PHX-9: not done (status=IN_REVIEW)
[Financial] severity=HIGH
    - approved_budget=60,533.77
    - forecast_spend=68,847.48
    - forecast_variance=-8,313.71
    - budget_consumption_pct=111.9%
    - actual_spend=67,749.24
    - approved_budget=60,533.77
    - remaining_budget=-81,700.31
    - approved_budget=60,533.77
    - actual_spend=67,749.24
    - committed_spend=74,484.84
    - budget_consumption_pct=111.9%
    - delivery_progress_pct=37.9

## A structural failure: simulated Jira outage

Distinguishes a HARD failure (`validation_passed=False`) from the soft data-quality degradations above (which only lower `confidence`).

In [8]:
from src.connectors.jira_client import JiraClient, JiraDataSource

class BrokenJiraSource(JiraDataSource):
    def fetch_issue_page(self, start_at, max_results, project_key=None, sprint_id=None):
        raise ConnectionError("simulated Jira outage")
    def fetch_issue_by_key(self, issue_key):
        return None

default_client = build_default_jira_client()
broken_client = JiraClient(source=BrokenJiraSource(), field_map=default_client.field_map, status_cfg=default_client.status_cfg)
broken_deps = NodeDeps(jira_client=broken_client, financial_source=deps.financial_source, memory_store=deps.memory_store, mapping=deps.mapping)
broken_graph = build_graph(broken_deps)

broken_result = broken_graph.invoke({
    "user_question": "portfolio status",
    "request_id": "req-003",
    "requested_at": datetime(2026, 9, 15, tzinfo=timezone.utc),
})
print(f"validation_passed: {broken_result['validation_passed']}  (False — a structural failure, not just a data-quality flag)")
print(f"confidence: {broken_result['confidence']}")
print(f"final_answer still produced: {bool(broken_result['final_answer'])}")
print()
print(broken_result["final_answer"][:300])

{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.570193+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.570472+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.02}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.571863+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.572758+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 0, "sprint_count": 0, "partial_failure": true}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.572926+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 0.95}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.574983+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}


{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.576505+00:00", "event": "graph_node_end", "node": "validate_delivery_data", "latency_ms": 1.43}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.578803+00:00", "event": "graph_node_start", "node": "fetch_financial_data"}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.579752+00:00", "event": "records_retrieved", "source": "Finance", "record_count": 6, "partial_failure": false}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.579886+00:00", "event": "graph_node_end", "node": "fetch_financial_data", "latency_ms": 0.94}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.581072+00:00", "event": "graph_node_start", "node": "validate_financial_data"}


{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.610855+00:00", "event": "graph_node_end", "node": "validate_financial_data", "latency_ms": 29.72}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.611557+00:00", "event": "graph_node_start", "node": "unify_projects"}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.612176+00:00", "event": "graph_node_end", "node": "unify_projects", "latency_ms": 0.58}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.612611+00:00", "event": "graph_node_start", "node": "retrieve_historical_memory"}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.613660+00:00", "event": "memory_retrieved", "project_count": 7, "snapshot_count": 7}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.613705+00:00", "event": "graph_node_end", "node": "retrieve_historical_memory", "latency_ms": 1.06}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.614129+00:00", "event": "graph_node_start", "node": "calc

{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.714796+00:00", "event": "graph_node_end", "node": "generate_response", "latency_ms": 65.6}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.715455+00:00", "event": "graph_node_start", "node": "persist_snapshot"}


{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.723152+00:00", "event": "datasource_access", "source": "Memory", "operation": "persist_snapshot", "written_count": 7}
{"request_id": "req-003", "timestamp": "2026-09-03T03:22:55.723295+00:00", "event": "graph_node_end", "node": "persist_snapshot", "latency_ms": 7.8}
validation_passed: False  (False — a structural failure, not just a data-quality flag)
confidence: LOW CONFIDENCE
final_answer still produced: True

# Weekly ELT Portfolio Report

## Data Freshness
Jira: 284.6 hours old
Finance: 284.6 hours old

## Executive Summary
7 tracked project(s) — 0 GREEN, 7 AMBER, 0 RED, 0 UNKNOWN.
Portfolio budget: $360,384.32  |  Spend: $347,611.89  |  Available: $-340,966.91
Projects requiring executive attention: Ph


Note the graph doesn't crash or halt on the outage — every project reads AMBER/UNKNOWN-leaning because `unify_projects` (which re-queries Jira independently) also fails for the delivery side, so `classify_financial_risk` is all that's left to inform `combine_risk`, which correctly refuses to call that GREEN on its own.

## Persisted snapshots

In [9]:
print(f"snapshot_persisted: {result['snapshot_persisted']}")
print(f"snapshots written this run: {len(result['snapshots_written'])}")
for sid in result["snapshots_written"][:3]:
    print(f"  {sid}")

snapshot_persisted: True
snapshots written this run: 7
  org-northwind__portfolio-elt-2026__PROJECT-10001#0
  org-northwind__portfolio-elt-2026__PROJECT-10002#0
  org-northwind__portfolio-elt-2026__PROJECT-10003#0


## Validation checks

- [x] Full graph compiles and executes all 14 nodes without raising
- [x] Portfolio-wide RAG distribution exactly matches Phase 1's original hand-computed prediction and every subsequent phase's reproduction of it
- [x] `project_filter` correctly narrows every downstream node (37 issues fetched, not 270)
- [x] `confidence` degrades honestly for real (not fabricated) staleness/reconciliation issues, without blocking a `final_answer`
- [x] A structural Jira outage sets `validation_passed=False`, distinct from the soft degradations above, and the graph still completes
- [x] `citations` correctly reflects only the sources actually consulted for the requested scope
- [x] One `ProjectSnapshot` persisted per project in scope

## Testing

`tests/test_graph.py` (29 tests) — every node factory unit-tested individually, plus 6 full end-to-end graph tests including the two-invocation history-accumulation test that `11_historical_analysis.ipynb` builds on next.

## Next step

`11_historical_analysis.ipynb` — invoke this same graph across several simulated weeks and watch real week-over-week trend classification and multi-sprint blocker detection emerge from the orchestrated flow, not from hand-assembled snapshots.